# 04 ? Mode Choice Join Preview and Analytical Dataset

This notebook first builds a local, stratified 350-trip preview to inspect the relationships among `viajes`, `personas`, `hogares`, vehicles, stages, and the ZAT distance table. It then uses the investigator-supplied classification to construct a full analytical dataset with one row per eligible trip.

The preview is exploratory and not representative. The full dataset retains source values, keeps both observed mode fields, and does not apply the expansion factor. Source records are never modified. Vehicle summaries are household-level attributes repeated on trip rows; they do not establish vehicle availability to a particular person or trip. Stage summaries remain preview-only.


In [ ]:
from pathlib import Path
import pandas as pd
from mode_choice.mode_choice_join_preview import build_join_preview

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Could not locate the project root.'
preview_path, summary = build_join_preview(PROJECT_ROOT, sample_size=350, random_seed=5082026)
preview = pd.read_csv(preview_path, dtype=str, keep_default_na=False)
assert len(preview) == summary['preview_rows'] == 350
assert preview['key_viaje'].nunique() == len(preview)
assert preview['persona__key_persona'].ne('').all()
assert preview['hogar__key_hg'].ne('').all()
assert sum(summary['sample_mode_counts'].values()) == 350
assert summary['distance_table_validation'] == {'row_count': 1475010, 'unique_directed_pairs': 1475010, 'zone_count': 1215}
assert summary['distance_status_counts'].get('unmatched_zone_pair', 0) == 0
assert summary['distance_status_counts'].get('missing_zone_code', 0) == 0
assert (preview.loc[preview.zat_distance_status == 'intrazonal_zero_derived', 'zat_centroid_distance_m'] == '0').all()
summary

In [ ]:
from IPython.display import display

preview_path = PROJECT_ROOT / 'data/interim/mode_choice_trip_join_preview_350.csv'
preview = pd.read_csv(preview_path, low_memory=False)
assert len(preview) == 350
print(f"Initial preview: {len(preview.head(10))} of {len(preview)} rows; {len(preview.columns)} columns")
display(preview.head(10))
column_profile = pd.DataFrame({
    'column': preview.columns,
    'inferred_type': [str(preview[column].dtype) for column in preview.columns],
    'missing_values': [int(preview[column].isna().sum()) for column in preview.columns],
    'distinct_values': [int(preview[column].nunique(dropna=True)) for column in preview.columns],
})
assert len(column_profile) == len(preview.columns) == 127
display(column_profile)


## Preview field descriptions and observed duplication

Exact copies, high match rates, and differences are reported separately. The 95% threshold is a practical flag only; it is not a statistical significance test and does not by itself justify dropping a field.


In [ ]:
field_dictionary_path = PROJECT_ROOT / 'docs/data_dictionary/mode_choice_trip_dataset_fields.csv'
field_dictionary = pd.read_csv(field_dictionary_path, dtype=str, keep_default_na=False)
field_dictionary = field_dictionary.set_index(['source_table', 'source_field'])

known_descriptions = {
    'key_hg': 'Technical household identifier.', 'key_pers': 'Person identifier linked to the trip.',
    'key_pers_viaja': 'Alternate identifier for the traveling person.', 'key_viaje': 'Unique technical trip identifier.',
    'zat_ori': 'Trip origin zone code.', 'zat_des': 'Trip destination zone code.',
    'zat_hg': 'Zone code associated with the household residence.',
    'modo_principal_agrupado': 'Observed grouped main travel mode.',
    'modo_principal_desagrupado': 'Observed detailed main travel mode.', 'fexp_vj': 'Trip expansion factor.',
}

def describe_preview_column(column):
    if column.startswith('persona__'):
        level, raw_name, table = 'person', column.removeprefix('persona__'), 'personas'
    elif column.startswith('hogar__'):
        level, raw_name, table = 'household', column.removeprefix('hogar__'), 'hogares'
    else:
        level, raw_name, table = 'trip', column, 'viajes'
    documented = ''
    if (table, raw_name) in field_dictionary.index:
        documented = field_dictionary.loc[(table, raw_name), 'documented_definition']
    if documented:
        return level, documented, 'Documented in local dictionary'
    if raw_name in known_descriptions:
        return level, known_descriptions[raw_name], 'Functional description; verify against survey documentation'
    return level, f"Field '{raw_name.replace('_', ' ')}' (description inferred from field name; confirm with survey documentation).", 'Inferred; official definition pending'

columns = list(preview.columns)
zat_columns = [column for column in ['zat_hg', 'zat_ori', 'zat_des', 'persona__zat_hg', 'hogar__zat_hg'] if column in preview]
zat_comparisons = []
for i, left in enumerate(zat_columns):
    for right in zat_columns[i + 1:]:
        comparable = preview[left].notna() & preview[right].notna() & preview[left].astype(str).str.strip().ne('') & preview[right].astype(str).str.strip().ne('')
        count = int(comparable.sum())
        matches = int(preview.loc[comparable, left].astype(str).eq(preview.loc[comparable, right].astype(str)).sum())
        zat_comparisons.append({'pair': f'{left} ? {right}', 'comparable_rows': count, 'matching_rows': matches,
                                'match_percent': round(100 * matches / count, 1) if count else None})

field_rows = []
for column in columns:
    level, description, definition_status = describe_preview_column(column)
    field_rows.append({'column': column, 'level/source': level, 'description_en': description,
                       'definition_status': definition_status,
                       'possible_repetition': 'See pairwise exact-copy and ZAT agreement checks below.'})
field_description_table = pd.DataFrame(field_rows)
assert len(field_description_table) == 127
display(field_description_table)
display(pd.DataFrame(zat_comparisons))
print('ZAT agreement is descriptive and does not imply conceptual equivalence. Percentages use comparable nonmissing rows only.')


## Relational structure

Keep one row per `key_viaje`. Link trips to people with `key_pers = key_persona` and to households with `key_hg`. Aggregate one-to-many vehicle records to household level before joining. Do not join raw stage rows directly to trips. Keep both observed mode fields as targets; they do not provide attributes for unchosen alternatives.


## Consolidated analytical dataset

Rebuild the one-row-per-trip dataset from the complete read-only relational source. Preserve every trip, source values, both observed mode outcomes, analytical keys, and `fexp_vj` without applying the expansion factor. Exact copies and investigator-directed exclusions are documented separately. Two unresolved fields remain in the export as pending candidates.

Retain the original origin and destination locality and municipality fields. Territory categories use `Localidad`, `Municipio`, and `No disponible`. Preserve `estra_hg` as categorical, including its original `No aplica` level. Do not filter for missingness, impute, recode source values, create time bands, perform EDA, or estimate models.


In [ ]:
import re
import sqlite3
from io import StringIO
from mode_choice.mode_choice_join_preview import _canonical_zone, _read_distance_table

classification_path = PROJECT_ROOT / 'docs/mode_choice_variable_classification.md'
classification_document = classification_path.read_text(encoding='utf-8')
classification_tsv = re.search(r'```tsv\s*\n(.*?)\n```', classification_document, flags=re.S)
assert classification_tsv is not None
classification = pd.read_csv(StringIO(classification_tsv.group(1)), sep='\t', dtype=str, keep_default_na=False)
classification['COLUMNA'] = classification['COLUMNA'].str.strip()
assert classification.shape == (136, 6) and classification['COLUMNA'].is_unique
assert classification['Decision Base Inicial'].iloc[:126].value_counts().to_dict() == {'NO': 66, 'Pending_duplicate_validation': 25, 'SI': 35}
assert classification['Final Decision'].isin(['INCLUDE','EXCLUDE','PENDING']).all()

database_path = PROJECT_ROOT / 'data/relational/em2023_relational.sqlite'
connection = sqlite3.connect(f'file:{database_path.as_posix()}?mode=ro', uri=True)
try:
    trips = pd.read_sql_query('SELECT * FROM "viajes" ORDER BY rowid', connection)
    people = pd.read_sql_query('SELECT * FROM "personas"', connection)
    households = pd.read_sql_query('SELECT * FROM "hogares"', connection)
    vehicles = pd.read_sql_query('SELECT * FROM "vehiculos"', connection)
    stages = pd.read_sql_query('SELECT * FROM "etapas"', connection)
finally:
    connection.close()
for source_frame in (trips, people, households, vehicles, stages):
    source_frame.columns = source_frame.columns.map(lambda name: name.strip() if isinstance(name, str) else name)

source_trip_rows = len(trips)
assert source_trip_rows == 100174
assert trips['key_viaje'].notna().all() and trips['key_viaje'].is_unique
assert people['key_persona'].notna().all() and people['key_persona'].is_unique
assert households['key_hg'].notna().all() and households['key_hg'].is_unique
assert {'nom_mun_ori','nom_mun_des','localidad_ori','localidad_des'}.issubset(trips.columns)

people_join = people.add_prefix('persona__')
households_join = households.add_prefix('hogar__')
full = trips.merge(people_join, left_on='key_pers', right_on='persona__key_persona', how='left', validate='many_to_one', indicator='_person_link')
unmatched_person_rows = int(full['_person_link'].eq('left_only').sum())
full = full.drop(columns='_person_link')
full = full.merge(households_join, left_on='key_hg', right_on='hogar__key_hg', how='left', validate='many_to_one', indicator='_household_link')
unmatched_household_rows = int(full['_household_link'].eq('left_only').sum())
full = full.drop(columns='_household_link')
assert unmatched_person_rows == 0 and unmatched_household_rows == 0
assert len(full) == source_trip_rows and full['key_viaje'].is_unique

vehicle_type_field = next(column for column in vehicles.columns if column.startswith('tipo_veh'))
vehicle_summary = vehicles.groupby('key_hg',as_index=False).agg(
    vehicle_records_in_household=('key_hg','size'),
    vehicle_type_values=(vehicle_type_field,lambda values:' | '.join(values.dropna().astype(str))))
full = full.merge(vehicle_summary,on='key_hg',how='left',validate='many_to_one')
full['vehicle_records_in_household'] = full['vehicle_records_in_household'].fillna(0).astype('int64')
vehicles['cantidad_numeric'] = pd.to_numeric(vehicles['cantidad'], errors='coerce')
vehicle_quantity = vehicles.groupby('key_hg', as_index=False)['cantidad_numeric'].sum().rename(columns={'cantidad_numeric':'vehicle_quantity_sum_exploratory'})
full = full.merge(vehicle_quantity, on='key_hg', how='left', validate='many_to_one')
full['vehicle_quantity_sum_exploratory'] = full['vehicle_quantity_sum_exploratory'].fillna(0)

stage_summary = stages.groupby('key_viaje',as_index=False).agg(
    stage_count=('key_etapa','size'),
    stage_mode_sequence=('modo_etapa',lambda values:' | '.join(values.dropna().astype(str))),
    stage_access_minutes_sum_exploratory=('t_acceso_min',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_wait_minutes_sum_exploratory=('t_espera_min',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_egress_minutes_sum_exploratory=('t_egreso_min',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_cost_sum_exploratory=('costo_viaje',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_payment_sum_exploratory=('pago_viaje',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_parking_cost_sum_exploratory=('costo_parqueo',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)))
full = full.merge(stage_summary,on='key_viaje',how='left',validate='one_to_one')
full['stage_count'] = full['stage_count'].fillna(0).astype('int64')

initial_pending_fields = classification.loc[classification['Decision Base Inicial'].eq('Pending_duplicate_validation'),'COLUMNA'].tolist()
classified_source_fields = set(classification.loc[classification['Decision Base Inicial'].ne('not in original classification'),'COLUMNA'])
assert classified_source_fields.issubset(set(full.columns) | {'zat_centroid_distance_m'}), f'Missing source fields: {sorted(classified_source_fields-set(full.columns)-{'zat_centroid_distance_m'})}'
print({'source_trip_rows':source_trip_rows,'joined_rows':len(full),'person_unmatched_rows':unmatched_person_rows,
       'household_unmatched_rows':unmatched_household_rows,'initial_classification_fields':126,
       'derived_classification_fields':10,'preview_only_unclassified_field':'zat_distance_status'})


from IPython.display import display
from pandas.api.types import is_numeric_dtype

analytical_csv_path = PROJECT_ROOT / 'data/analytical/em2023_mode_choice_analytical_dataset.csv'
existing_export_columns = pd.read_csv(analytical_csv_path, nrows=0).columns.tolist()
existing_trip_keys = pd.read_csv(analytical_csv_path, usecols=['key_viaje'], dtype={'key_viaje': 'string'}, low_memory=False)['key_viaje']
assert len(existing_export_columns) == 49
assert len(existing_trip_keys) == source_trip_rows == 100174
assert existing_trip_keys.notna().all() and existing_trip_keys.is_unique

person_expansion_columns = [column for column in full.columns if column.startswith('persona__fexp_per>5a')]
assert len(person_expansion_columns) == 1, f'Expected one person expansion-factor field, found {person_expansion_columns}'
person_expansion_column = person_expansion_columns[0]

numeric_field_specs = [
    ('hora_ini', 'decimal', False),
    ('hora_fin', 'decimal', False),
    ('duracion_min', 'decimal', False),
    ('t_acceso_min', 'decimal', True),
    ('t_espera_min', 'decimal', True),
    ('t_egreso_min', 'decimal', True),
    ('fexp_vj', 'colombian', False),
    (person_expansion_column, 'colombian', False),
]
assert all(column in full.columns for column, _, _ in numeric_field_specs)


def parse_numeric_field(series, field_name, locale, dash_is_missing=False):
    raw = series.astype('string').str.strip()
    source_missing = series.isna() | raw.eq('')
    dash_mask = raw.eq('-')
    expected_missing = source_missing | (dash_mask if dash_is_missing else False)
    candidate = raw.mask(expected_missing)

    if locale == 'colombian':
        valid_pattern = r'[+-]?(?:\d+|\d{1,3}(?:\.\d{3})+)(?:,\d+)?'
        valid = candidate.str.fullmatch(valid_pattern, na=False)
        grouped_thousands = candidate.str.fullmatch(r'[+-]?\d{1,3}(?:\.\d{3})+(?:,\d+)?', na=False)
        normalized = candidate.where(~grouped_thousands, candidate.str.replace('.', '', regex=False))
        normalized = normalized.str.replace(',', '.', regex=False)
    else:
        valid = candidate.str.fullmatch(r'[+-]?\d+(?:[,.]\d+)?', na=False)
        normalized = candidate.str.replace(',', '.', regex=False)

    parsed = pd.to_numeric(normalized.where(valid), errors='coerce').astype('Float64')
    unparseable = candidate.notna() & (~valid | parsed.isna())
    audit = {
        'field': field_name,
        'locale_rule': locale,
        'rows': len(series),
        'source_missing': int(source_missing.sum()),
        'dash_to_missing': int(dash_mask.sum()) if dash_is_missing else 0,
        'numeric_values': int(parsed.notna().sum()),
        'unparseable_values': int(unparseable.sum()),
    }
    unexpected = raw.loc[unparseable].drop_duplicates().head(20).tolist()
    return parsed, audit, unexpected, dash_mask


# Guard the two critical locale interpretations independently of the source data.
assert parse_numeric_field(pd.Series(['6,50']), 'example_decimal_hour', 'decimal')[0].iloc[0] == 6.5
assert parse_numeric_field(pd.Series(['1.364,7']), 'example_expansion_factor', 'colombian')[0].iloc[0] == 1364.7

numeric_type_audit = []
unexpected_numeric_values = {}
dash_masks = {}
for field, locale, dash_is_missing in numeric_field_specs:
    original = full[field].copy()
    parsed, audit, unexpected, dash_mask = parse_numeric_field(
        original, field, locale, dash_is_missing=dash_is_missing
    )
    full[field] = parsed
    numeric_type_audit.append(audit)
    dash_masks[field] = dash_mask
    if unexpected:
        unexpected_numeric_values[field] = unexpected

display(pd.DataFrame(numeric_type_audit))
if unexpected_numeric_values:
    print({'unparseable_values_by_field': unexpected_numeric_values})
    raise ValueError('Numeric normalization stopped: unexpected source values require review before export.')

normalized_numeric_fields = [column for column, _, _ in numeric_field_specs]
assert all(is_numeric_dtype(full[column].dtype) for column in normalized_numeric_fields)
for field, _, dash_is_missing in numeric_field_specs:
    assert str(full[field].dtype) == 'Float64'
    if dash_is_missing:
        assert full.loc[dash_masks[field], field].isna().all(), f'Dashes must become missing for {field}'
        assert not full.loc[dash_masks[field], field].eq(0).any(), f'Dashes must never become zero for {field}'

CSV_IMPORT_DTYPES = {
    **{column: 'Float64' for column in normalized_numeric_fields},
    **{column: 'string' for column in [
        'key_hg', 'key_pers', 'key_pers_viaja', 'key_viaje',
        'zat_ori', 'zat_des', 'hogar__cod_upl_hg',
        'modo_principal_agrupado', 'modo_principal_desagrupado',
        'motivo_viaje', 'frecuencia_viaje', 'app_antes_vj', 'max_nivel_edu',
        'estra_hg', 'persona__licencia_cond_vigente',
        'start_time_interpreted', 'end_time_interpreted',
    ]},
    'start_minutes_after_midnight': 'Int64',
    'end_minutes_after_midnight': 'Int64',
    'duration_calculated_min': 'Int64',
    'crosses_midnight': 'boolean',
}
print({'numeric_fields_normalized': normalized_numeric_fields,
       'pandas_numeric_dtype': 'Float64',
       'csv_category_and_identifier_import_dtypes': 'string'})


In [ ]:
from IPython.display import display
from pandas.api.types import is_numeric_dtype

analytical_csv_path = PROJECT_ROOT / 'data/analytical/em2023_mode_choice_analytical_dataset.csv'
existing_export_columns = pd.read_csv(analytical_csv_path, nrows=0).columns.tolist()
existing_trip_keys = pd.read_csv(analytical_csv_path, usecols=['key_viaje'], dtype={'key_viaje': 'string'}, low_memory=False)['key_viaje']
assert len(existing_export_columns) == 49
assert len(existing_trip_keys) == source_trip_rows == 100174
assert existing_trip_keys.notna().all() and existing_trip_keys.is_unique

person_expansion_columns = [column for column in full.columns if column.startswith('persona__fexp_per>5a')]
assert len(person_expansion_columns) == 1, f'Expected one person expansion-factor field, found {person_expansion_columns}'
person_expansion_column = person_expansion_columns[0]

numeric_field_specs = [
    ('hora_ini', 'decimal', True),
    ('hora_fin', 'decimal', True),
    ('duracion_min', 'decimal', False),
    ('t_acceso_min', 'decimal', True),
    ('t_espera_min', 'decimal', True),
    ('t_egreso_min', 'decimal', True),
    ('fexp_vj', 'colombian', False),
    (person_expansion_column, 'colombian', False),
]
assert all(column in full.columns for column, _, _ in numeric_field_specs)


def parse_numeric_field(series, field_name, locale, dash_is_missing=False):
    raw = series.astype('string').str.strip()
    source_missing = series.isna() | raw.eq('')
    dash_mask = raw.eq('-')
    expected_missing = source_missing | (dash_mask if dash_is_missing else False)
    candidate = raw.mask(expected_missing)

    if locale == 'colombian':
        valid_pattern = r'[+-]?(?:\d+|\d{1,3}(?:\.\d{3})+)(?:,\d+)?'
        valid = candidate.str.fullmatch(valid_pattern, na=False)
        grouped_thousands = candidate.str.fullmatch(r'[+-]?\d{1,3}(?:\.\d{3})+(?:,\d+)?', na=False)
        normalized = candidate.where(~grouped_thousands, candidate.str.replace('.', '', regex=False))
        normalized = normalized.str.replace(',', '.', regex=False)
    else:
        valid = candidate.str.fullmatch(r'[+-]?\d+(?:[,.]\d+)?', na=False)
        normalized = candidate.str.replace(',', '.', regex=False)

    parsed = pd.to_numeric(normalized.where(valid), errors='coerce').astype('Float64')
    unparseable = candidate.notna() & (~valid | parsed.isna())
    audit = {
        'field': field_name,
        'locale_rule': locale,
        'rows': len(series),
        'source_missing': int(source_missing.sum()),
        'dash_to_missing': int(dash_mask.sum()) if dash_is_missing else 0,
        'numeric_values': int(parsed.notna().sum()),
        'unparseable_values': int(unparseable.sum()),
    }
    unexpected = raw.loc[unparseable].drop_duplicates().head(20).tolist()
    return parsed, audit, unexpected, dash_mask


# Guard the two critical locale interpretations independently of the source data.
assert parse_numeric_field(pd.Series(['6,50']), 'example_decimal_hour', 'decimal')[0].iloc[0] == 6.5
assert parse_numeric_field(pd.Series(['1.364,7']), 'example_expansion_factor', 'colombian')[0].iloc[0] == 1364.7

numeric_type_audit = []
unexpected_numeric_values = {}
dash_masks = {}
for field, locale, dash_is_missing in numeric_field_specs:
    original = full[field].copy()
    parsed, audit, unexpected, dash_mask = parse_numeric_field(
        original, field, locale, dash_is_missing=dash_is_missing
    )
    full[field] = parsed
    numeric_type_audit.append(audit)
    dash_masks[field] = dash_mask
    if unexpected:
        unexpected_numeric_values[field] = unexpected

display(pd.DataFrame(numeric_type_audit))
if unexpected_numeric_values:
    print({'unparseable_values_by_field': unexpected_numeric_values})
    raise ValueError('Numeric normalization stopped: unexpected source values require review before export.')

normalized_numeric_fields = [column for column, _, _ in numeric_field_specs]
assert all(is_numeric_dtype(full[column].dtype) for column in normalized_numeric_fields)
for field, _, dash_is_missing in numeric_field_specs:
    assert str(full[field].dtype) == 'Float64'
    if dash_is_missing:
        assert full.loc[dash_masks[field], field].isna().all(), f'Dashes must become missing for {field}'
        assert not full.loc[dash_masks[field], field].eq(0).any(), f'Dashes must never become zero for {field}'

CSV_IMPORT_DTYPES = {
    **{column: 'Float64' for column in normalized_numeric_fields},
    **{column: 'string' for column in [
        'key_hg', 'key_pers', 'key_pers_viaja', 'key_viaje',
        'zat_ori', 'zat_des', 'hogar__cod_upl_hg',
        'modo_principal_agrupado', 'modo_principal_desagrupado',
        'motivo_viaje', 'frecuencia_viaje', 'app_antes_vj', 'max_nivel_edu',
        'estra_hg', 'persona__licencia_cond_vigente',
        'start_time_interpreted', 'end_time_interpreted',
    ]},
    'start_minutes_after_midnight': 'Int64',
    'end_minutes_after_midnight': 'Int64',
    'duration_calculated_min': 'Int64',
    'crosses_midnight': 'boolean',
}
print({'numeric_fields_normalized': normalized_numeric_fields,
       'pandas_numeric_dtype': 'Float64',
       'csv_category_and_identifier_import_dtypes': 'string'})


In [ ]:
pending_fields = initial_pending_fields
explicit_counterparts = {'persona__cod_per':'cod_pers','persona__cod_hg':'cod_hg','persona__key_hg':'key_hg',
                         'persona__key_persona':'key_pers','hogar__cod_hog':'cod_hg'}
classification_by_field = classification.set_index('COLUMNA').to_dict('index')
explicit_exclusion_fields = {'persona__actividad_economica','hogar__cod_viv','hogar__cod_hog','hogar__nom_utam_hg'}
pending_comparisons=[]
for field in pending_fields:
    if field in explicit_counterparts:
        counterpart=explicit_counterparts[field]
    elif field.startswith('persona__'):
        candidate=field.removeprefix('persona__'); counterpart=candidate if candidate in trips.columns else None
    elif field.startswith('hogar__'):
        candidate=field.removeprefix('hogar__'); counterpart=candidate if candidate in trips.columns else None
    else:
        counterpart=None
    if counterpart is None or counterpart not in full.columns:
        comparison={'pending_field':field,'counterpart_field':'','comparison_level':'','linked_trip_rows':len(full),
          'comparable_rows':0,'equal_rows':0,'match_percent':None,'different_nonmissing_rows':0,'difference_percent':None,
          'pending_missing_rows':int(full[field].isna().sum()),'counterpart_missing_rows':None,
          'pending_only_missing_rows':None,'counterpart_only_missing_rows':None,'both_missing_rows':None,
          'coverage_percent':0.0,'possible_duplicate_relevant':False,'exact_value_copy':False}
    else:
        left,right=full[field],full[counterpart]
        left_missing=left.isna()|left.astype('string').str.strip().eq('')
        right_missing=right.isna()|right.astype('string').str.strip().eq('')
        comparable=~left_missing&~right_missing
        equal=left.loc[comparable].astype(str).eq(right.loc[comparable].astype(str))
        n=int(comparable.sum()); matches=int(equal.sum()); differences=n-matches
        rate=100*matches/n if n else None
        left_only=int((left_missing&~right_missing).sum()); right_only=int((right_missing&~left_missing).sum())
        both_missing=int((left_missing&right_missing).sum())
        exact=n>0 and differences==0 and left_only==0 and right_only==0
        comparison={'pending_field':field,'counterpart_field':counterpart,
          'comparison_level':'trip-person' if field.startswith('persona__') else 'trip-household',
          'linked_trip_rows':len(full),'comparable_rows':n,'equal_rows':matches,
          'match_percent':round(rate,3) if rate is not None else None,'different_nonmissing_rows':differences,
          'difference_percent':round(100*differences/n,3) if n else None,
          'pending_missing_rows':int(left_missing.sum()),'counterpart_missing_rows':int(right_missing.sum()),
          'pending_only_missing_rows':left_only,'counterpart_only_missing_rows':right_only,'both_missing_rows':both_missing,
          'coverage_percent':round(100*n/len(full),3),'possible_duplicate_relevant':rate is not None and rate>=95,
          'exact_value_copy':exact}
    if field in explicit_exclusion_fields:
        decision=classification_by_field[field]['Final Rationale']
    elif comparison['exact_value_copy']:
        decision=f"exclude exact copy of {counterpart}; retain the canonical selected representation where applicable"
    else:
        decision=classification_by_field[field]['Final Rationale']
    comparison['decision']=decision
    comparison['final_decision']=classification_by_field[field]['Final Decision']
    pending_comparisons.append(comparison)
duplicate_validation=pd.DataFrame(pending_comparisons)
exact_copy_fields=set(duplicate_validation.loc[duplicate_validation.exact_value_copy,'pending_field'])
assert len(duplicate_validation)==25 and len(exact_copy_fields)==20
assert int(duplicate_validation.final_decision.eq('PENDING').sum())==2
display(duplicate_validation)
print({'pending_fields_tested':len(pending_fields),'exact_copies_excluded':len(exact_copy_fields),
       'remaining_pending_fields':int(duplicate_validation.final_decision.eq('PENDING').sum()),
       'fields_without_direct_counterpart':int(duplicate_validation.counterpart_field.eq('').sum())})


In [ ]:
import numpy as np

# Interpret decimal-hour fields without changing their original source columns.
def decimal_hour_minutes(series):
    values=pd.to_numeric(series.astype('string').str.strip().str.replace(',', '.', regex=False),errors='coerce')
    return np.rint(values*60).astype('Int64')

start_minutes=decimal_hour_minutes(full['hora_ini'])
end_minutes=decimal_hour_minutes(full['hora_fin'])
reported_duration=pd.to_numeric(full['duracion_min'].astype('string').str.strip().str.replace(',', '.', regex=False),errors='coerce')
full['start_minutes_after_midnight']=start_minutes
full['end_minutes_after_midnight']=end_minutes
full['start_time_interpreted']=start_minutes.map(lambda value:f'{int(value)//60:02d}:{int(value)%60:02d}' if pd.notna(value) else pd.NA)
full['end_time_interpreted']=end_minutes.map(lambda value:f'{int(value)//60:02d}:{int(value)%60:02d}' if pd.notna(value) else pd.NA)
full['duration_calculated_min']=((end_minutes-start_minutes)%1440).astype('Int64')
full['crosses_midnight']=(end_minutes<start_minutes).where(start_minutes.notna()&end_minutes.notna(),pd.NA).astype('boolean')

def valid_territory(value):
    if pd.isna(value): return False
    text=str(value).strip()
    return bool(text) and text.casefold()!='no aplica'

def territory_name(locality,municipality):
    if valid_territory(locality): return locality
    if valid_territory(municipality): return municipality
    return 'No disponible'

def territory_type(locality,municipality):
    if valid_territory(locality): return 'Localidad'
    if valid_territory(municipality): return 'Municipio'
    return 'No disponible'

for side in ['ori','des']:
    locality=full[f'localidad_{side}']
    municipality=full[f'nom_mun_{side}']
    full[f'territory_name_{side}']=[territory_name(l,m) for l,m in zip(locality,municipality)]
    full[f'territory_type_{side}']=[territory_type(l,m) for l,m in zip(locality,municipality)]

origin=full['zat_ori'].map(lambda value:_canonical_zone(value,'zat_ori'))
destination=full['zat_des'].map(lambda value:_canonical_zone(value,'zat_des'))
needed_pairs={(int(a),int(b)) for a,b in zip(origin,destination) if a is not None and b is not None and a!=b}
distance_path=PROJECT_ROOT/'data/sources/raw/zonification/ZAT_Distance.csv'
distance_lookup,distance_validation=_read_distance_table(distance_path,needed_pairs)
full['zat_centroid_distance_m']=[0 if a is not None and b is not None and a==b else distance_lookup.get((a,b),pd.NA) if a is not None and b is not None else pd.NA for a,b in zip(origin,destination)]

geo_rows=[]
for side in ['ori','des']:
    locality=full[f'localidad_{side}']; municipality=full[f'nom_mun_{side}']
    locality_valid=locality.map(valid_territory)
    municipality_valid=municipality.map(valid_territory)
    fallback=(~locality_valid)&municipality_valid
    unavailable=(~locality_valid)&(~municipality_valid)
    used_municipalities=sorted(set(municipality.loc[fallback].dropna().astype(str)))
    after=full[f'territory_type_{side}'].value_counts(dropna=False).to_dict()
    geo_rows.append({'direction':'origin' if side=='ori' else 'destination','source_rows':len(full),
      'locality_valid_before':int(locality_valid.sum()),
      'locality_no_aplica_before':int(locality.astype('string').str.strip().str.casefold().eq('no aplica').sum()),
      'municipality_fallback_rows':int(fallback.sum()),'unidentified_before':int(unavailable.sum()),
      'territory_localidad_after':int(after.get('Localidad',0)),
      'territory_municipio_after':int(after.get('Municipio',0)),
      'territory_no_disponible_after':int(after.get('No disponible',0)),
      'municipalities_used':' | '.join(used_municipalities),'unique_municipality_count':len(used_municipalities)})
territory_counts=pd.DataFrame(geo_rows)
assert (territory_counts.source_rows==territory_counts[['territory_localidad_after','territory_municipio_after','territory_no_disponible_after']].sum(axis=1)).all()
assert territory_counts.territory_no_disponible_after.sum()==0
display(territory_counts)

output_classification=classification[classification['Final Decision'].isin(['INCLUDE','PENDING'])]
output_columns=output_classification['COLUMNA'].tolist()
assert len(output_columns)==len(set(output_columns)) and set(output_columns).issubset(full.columns)
assert 'zat_distance_status' not in output_columns
analytical=full.loc[:,output_columns].copy()
assert len(analytical)==source_trip_rows and analytical.key_viaje.notna().all() and analytical.key_viaje.is_unique
required={'modo_principal_agrupado','modo_principal_desagrupado','key_viaje','key_pers','key_hg','fexp_vj',
          'nom_mun_ori','nom_mun_des','territory_name_ori','territory_type_ori','territory_name_des','territory_type_des'}
assert required.issubset(analytical.columns)
assert not {'persona__actividad_economica','hogar__cod_viv','hogar__cod_hog','hogar__nom_utam_hg'}.intersection(analytical.columns)
assert not exact_copy_fields.intersection(analytical.columns)


## Export and round-trip validation

Validate the unchanged 49-column schema and trip keys before writing. Stage the CSV temporarily, reload it using `CSV_IMPORT_DTYPES`, validate numeric and clock-time values, then atomically replace only the existing analytical CSV. CSV cannot embed dtypes, so downstream pandas reads should reuse the import map above.


In [ ]:
from pathlib import Path
import os, tempfile

derived_definitions={
 'zat_centroid_distance_m':('Distance lookup','trip OD pair','float64','predictor','Distance between origin and destination ZAT centroids in meters; intrazonal pairs are zero.'),
 'start_time_interpreted':('viajes','trip','string','derived_time','Interpreted HH:MM value from decimal-hour source.'),
 'end_time_interpreted':('viajes','trip','string','derived_time','Interpreted HH:MM value from decimal-hour source.'),
 'start_minutes_after_midnight':('viajes','trip','Int64','derived_time','Decimal-hour value rounded after conversion to minutes.'),
 'end_minutes_after_midnight':('viajes','trip','Int64','derived_time','Decimal-hour value rounded after conversion to minutes.'),
 'duration_calculated_min':('viajes','trip','Int64','derived_time','End minus start modulo 24 hours.'),
 'crosses_midnight':('viajes','trip','boolean','quality_time','True when interpreted end time is earlier than interpreted start time.'),
 'territory_name_ori':('viajes','trip','string','territorial_context','Origin locality when valid; otherwise municipality; otherwise No disponible.'),
 'territory_type_ori':('viajes','trip','string','territorial_context','Origin territory type: Localidad, Municipio, or No disponible.'),
 'territory_name_des':('viajes','trip','string','territorial_context','Destination locality when valid; otherwise municipality; otherwise No disponible.'),
 'territory_type_des':('viajes','trip','string','territorial_context','Destination territory type: Localidad, Municipio, or No disponible.'),
 'vehicle_records_in_household':('vehiculos','household','int64','household_context','Count of vehicle records in the household; not person-level availability.'),
 'vehicle_type_values':('vehiculos','household','object','household_context','Vehicle types recorded for the household; not person-level availability.'),
}
source_dictionary=pd.read_csv(PROJECT_ROOT/'docs/data_dictionary/mode_choice_trip_dataset_fields.csv',dtype=str,keep_default_na=False)
source_definitions={(row.source_table,row.source_field):row.documented_definition for row in source_dictionary.itertuples()}
field_rows=[]
for row in classification.to_dict('records'):
    column=row['COLUMNA']; included=column in analytical.columns
    if column in derived_definitions:
        table,level,dtype,role,description=derived_definitions[column]
        source_field=''; transformation=description
        missing=int(analytical[column].isna().sum()) if included else int(full[column].isna().sum())
        limitation='Derived field; source fields remain available in their original columns.'
    else:
        if column.startswith('persona__'):
            table,source_field,level='personas',column.removeprefix('persona__'),'person'
        elif column.startswith('hogar__'):
            table,source_field,level='hogares',column.removeprefix('hogar__'),'household'
        elif column.startswith('stage_'):
            table,source_field,level='etapas',column,'trip aggregation'
        elif column.startswith('vehicle_'):
            table,source_field,level='vehiculos',column,'household aggregation'
        else:
            table,source_field,level='viajes',column,'trip'
        dtype=str(full[column].dtype); role=row['Possible research_role']
        description=source_definitions.get((table,source_field),'') or f'Source field {source_field}; official survey definition is unavailable in the local dictionary.'
        transformation='Joined by validated technical key; source value retained.' if level!='trip' else 'Source value retained without recoding.'
        missing=int(full[column].isna().sum()); limitation=''
    if row['Final Decision']=='PENDING': limitation='Retained in the export pending researcher review; not approved as a model predictor.'
    if row['Final Decision']=='EXCLUDE': limitation=row['Final Rationale']
    field_rows.append({'column':column,'source_table':table,'source_field':source_field,'level':level,'data_type':dtype,
      'research_role':row['Possible research_role'],'initial_decision':row['Decision Base Inicial'],
      'final_decision':row['Final Decision'],'included_in_export':included,'final_rationale':row['Final Rationale'],
      'description_en':description,'transformation':transformation,'missing_values':missing,'limitations':limitation})
field_dictionary_export=pd.DataFrame(field_rows)
assert field_dictionary_export.loc[field_dictionary_export.included_in_export,'column'].tolist()==analytical.columns.tolist()

for index,row in duplicate_validation.iterrows():
    field=row['pending_field']
    duplicate_validation.loc[index,'final_decision']=classification_by_field[field]['Final Decision']
    if field in explicit_exclusion_fields:
        duplicate_validation.loc[index,'decision']=classification_by_field[field]['Final Rationale']
    elif bool(row['exact_value_copy']):
        duplicate_validation.loc[index,'decision']=f"exclude exact copy of {row['counterpart_field']}; retain the canonical selected representation where applicable"
    else:
        duplicate_validation.loc[index,'decision']=classification_by_field[field]['Final Rationale']

complete_times=start_minutes.notna()&end_minutes.notna()&reported_duration.notna()
duration_difference=(full.loc[complete_times,'duration_calculated_min'].astype(float)-reported_duration.loc[complete_times]).abs()
estra_counts=full['estra_hg'].astype('string').value_counts(dropna=False).to_dict()
quality_rows=[
 {'metric':'source_trip_rows','value':source_trip_rows,'detail':'All source viajes records'},
 {'metric':'analytical_rows','value':len(analytical),'detail':'No records removed for predictor missingness'},
 {'metric':'analytical_columns','value':len(analytical.columns),'detail':'Included fields match dictionary rows marked included'},
 {'metric':'key_viaje_unique_nonnull','value':bool(analytical.key_viaje.notna().all() and analytical.key_viaje.is_unique),'detail':'One row per trip'},
 {'metric':'person_unmatched_rows','value':unmatched_person_rows,'detail':'Validated many-to-one person join'},
 {'metric':'household_unmatched_rows','value':unmatched_household_rows,'detail':'Validated many-to-one household join'},
 {'metric':'duplicate_pending_fields_reviewed','value':len(duplicate_validation),'detail':'All original pending fields appear in the matrix'},
 {'metric':'exact_duplicate_fields_removed','value':len(exact_copy_fields),'detail':'All exact copies removed from analytical export'},
 {'metric':'explicit_researcher_exclusions','value':4,'detail':'Four specified fields excluded; activity is not treated as a duplicate of occupation'},
 {'metric':'remaining_pending_fields_in_export','value':int(duplicate_validation.final_decision.eq('PENDING').sum()),'detail':'Retained for later review; not approved as predictors'},
 {'metric':'expansion_factor_applied','value':False,'detail':'fexp_vj retained as source variable'},
 {'metric':'estrato_no_aplica_count','value':int(estra_counts.get('No aplica',0)),'detail':'Original value preserved as a categorical level'},
 {'metric':'estrato_category_counts','value':str(estra_counts),'detail':'Original estra_hg categories; no numeric recoding'},
 {'metric':'complete_time_records','value':int(complete_times.sum()),'detail':'Start, end, and reported duration available'},
 {'metric':'time_records_with_any_missing_component','value':int((~complete_times).sum()),'detail':'Retained in the output'},
 {'metric':'duration_exact_matches','value':int((duration_difference==0).sum()),'detail':'Calculated versus reported duration'},
 {'metric':'duration_differences_over_one_minute','value':int((duration_difference>1).sum()),'detail':'Calculated versus reported duration'},
 {'metric':'trips_crossing_midnight','value':int(full.crosses_midnight.fillna(False).sum()),'detail':'End time earlier than start time'},
 {'metric':'time_of_day_bands_created','value':False,'detail':'Time bands remain for EDA'},
 {'metric':'ZAT_distance_lookup_validation','value':str(distance_validation),'detail':'Distance table validation'},
 {'metric':'ZAT_distance_missing','value':int(analytical.zat_centroid_distance_m.isna().sum()),'detail':'Missing OD zone or unmatched distance lookup'},
]
for geo in territory_counts.to_dict('records'):
    direction=geo['direction']
    for key,value in geo.items():
        if key!='direction': quality_rows.append({'metric':f'territory_{direction}_{key}','value':value,'detail':'Territorial harmonization before/after counts and municipality coverage'})
for column in analytical.columns:
    quality_rows.append({'metric':f'missing::{column}','value':int(analytical[column].isna().sum()),'detail':'Exported field missingness'})
for column in ['modo_principal_agrupado','modo_principal_desagrupado']:
    for category,count in analytical[column].value_counts(dropna=False).items():
        quality_rows.append({'metric':f'distribution::{column}::{category}','value':int(count),'detail':'Observed outcome category count'})
quality_report=pd.DataFrame(quality_rows)

dataset_path = PROJECT_ROOT / 'data/analytical/em2023_mode_choice_analytical_dataset.csv'
assert len(analytical) == source_trip_rows == 100174
assert len(analytical.columns) == 49
assert analytical.columns.tolist() == existing_export_columns, 'The analytical schema or column order changed.'
assert analytical['key_viaje'].notna().all() and analytical['key_viaje'].is_unique
assert analytical['key_viaje'].astype('string').reset_index(drop=True).equals(existing_trip_keys.reset_index(drop=True)), 'Trip records or order changed.'
assert all(is_numeric_dtype(analytical[column].dtype) for column in normalized_numeric_fields)
assert all(str(analytical[column].dtype) == 'Float64' for column in normalized_numeric_fields)

# Validate the candidate CSV before replacing the existing analytical dataset.
staged_path = None
try:
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8-sig', newline='', suffix='.tmp',
                                     dir=dataset_path.parent, delete=False) as stream:
        staged_path = Path(stream.name)
        analytical.to_csv(stream, index=False)

    candidate = pd.read_csv(staged_path, dtype=CSV_IMPORT_DTYPES, low_memory=False)
    assert candidate.shape == (100174, 49)
    assert candidate.columns.tolist() == existing_export_columns
    assert candidate['key_viaje'].notna().all() and candidate['key_viaje'].is_unique
    assert candidate['key_viaje'].reset_index(drop=True).equals(existing_trip_keys.reset_index(drop=True))
    for field in normalized_numeric_fields:
        assert str(candidate[field].dtype) == 'Float64'
        assert candidate[field].notna().sum() == analytical[field].notna().sum()
        pd.testing.assert_series_equal(candidate[field], analytical[field].reset_index(drop=True), check_names=False)
    for field in ['start_time_interpreted', 'end_time_interpreted']:
        valid_clock = candidate[field].isna() | candidate[field].str.fullmatch(r'(?:[01]\d|2[0-3]):[0-5]\d', na=False)
        assert valid_clock.all(), f'Invalid HH:MM values in {field}'
    for field, _, dash_is_missing in numeric_field_specs:
        if dash_is_missing:
            assert int(candidate[field].isna().sum()) == int(full[field].isna().sum())
            assert full.loc[dash_masks[field], field].isna().all()

    os.replace(staged_path, dataset_path)
    staged_path = None
finally:
    if staged_path is not None:
        staged_path.unlink(missing_ok=True)

# Reload the canonical path and verify the written artifact, including its CSV dtype map.
reloaded = pd.read_csv(dataset_path, dtype=CSV_IMPORT_DTYPES, low_memory=False)
assert reloaded.shape == (100174, 49)
assert reloaded.columns.tolist() == existing_export_columns
assert reloaded['key_viaje'].notna().all() and reloaded['key_viaje'].is_unique
assert reloaded['key_viaje'].reset_index(drop=True).equals(existing_trip_keys.reset_index(drop=True))
assert all(str(reloaded[field].dtype) == 'Float64' for field in normalized_numeric_fields)
assert reloaded['zat_ori'].dtype == 'string' and reloaded['zat_des'].dtype == 'string'
assert reloaded['start_time_interpreted'].dtype == 'string'
assert reloaded['end_time_interpreted'].dtype == 'string'
print({'export_path': str(dataset_path), 'rows': len(reloaded), 'columns': len(reloaded.columns),
       'normalized_numeric_dtypes': {field: str(reloaded[field].dtype) for field in normalized_numeric_fields},
       'dash_values_converted_to_missing': {field: int(dash_masks[field].sum()) for field, _, dash_is_missing in numeric_field_specs if dash_is_missing},
       'unparseable_source_values': 0,
       'interpreted_time_examples': reloaded[['start_time_interpreted','end_time_interpreted']].head(3).to_dict('records'),
       'csv_import_dtype_map': CSV_IMPORT_DTYPES})
